**Normalization parameters by age**

Note: let $x$ be age (years). For RR mean, RR standard deviation, and pNN50 use the formulas below.

RR mean:

$$
\text{RR\_mean} = 505 \cdot x^{0.122}
$$

For RR standard deviation and pNN50 use the age-dependent formulas:

If $x \le 12$:

$$
\text{RR\_std} = 80 \cdot x^{0.26} \\,
\text{pNN50} = 0.037 \cdot x^{0.78}
$$

If $x > 12$:

$$
\text{RR\_std} = 290 \cdot x^{-0.2} \\,
\text{pNN50} = 5 \cdot x^{-1.1}
$$

Python implementation:

```python
# x = age (years)
RR_mean = 505 * x**0.122
if x <= 12:
    RR_std = 80 * x**0.26
    pNN50 = 0.037 * x**0.78
else:
    RR_std = 290 * x**(-0.2)
    pNN50 = 5 * x**(-1.1)
```

In [1]:
import numpy as np
from itertools import chain
import pandas as pd
import json
from utils import normalize, extract_hrv_features
import os
import h5py
from pathlib import Path

ages_table_path = 'ages_table.xlsx'
ages_table = pd.read_excel("ages_table.xlsx", dtype={"code": "string"})
ages_table["code"] = ages_table["code"].str.strip()
subjects_path = 'series'

with open("subjects_train.json", "r", encoding="utf-8") as f:
    subjects_train = json.load(f)

with open("subjects_test.json", "r", encoding="utf-8") as f:
    subjects_test = json.load(f)

feature_cols = ['mean', 'sdsd', 'sd2', 'ccm', 'guzik', 'nn50', 'porta', 'std']
rr_cols = [f'rr_{i}' for i in range(1, 21)]
rr_last_col = 'rr_20'

I0000 00:00:1785248636.644005   47554 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1785248639.083391   47554 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
# ------------------------------------------------------------
# CONFIG
# ------------------------------------------------------------
OUTPUT_H5 = "hrv_dataset.h5"
YEAR_TO_WEEKS = 52.14
EPS = 1e-8

# Features in your dataframe
interval_norm_cols = ["sdsd", "sd2", "ccm", "guzik", "nn50", "porta", "std", "target"]
subject_specific_cols = ["mean"] + [f"rr_{i}" for i in range(1, 21)]

# What will be saved as X and y
x_cols = [c for c in interval_norm_cols if c != "target"] + subject_specific_cols
y_col = "target"

# If your subject codes need leading zeros, format them here.
# Example: f"{int(subj):03d}" for 009.txt
def subject_to_filename(subj):
    if len(str(subj)) < 3:
        return f"{int(subj):03d}.txt"
    else:
        return f"{str(subj)}.txt"


# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def get_age_years(subj, ages_table, year_to_weeks=52.14):

    age_weeks = ages_table.loc[ages_table["code"] == subj, "age-weeks"].values
    if len(age_weeks) == 0:
        raise ValueError(f"Subject {subj} not found in ages_table.")
    return float(age_weeks[0]) / year_to_weeks


def get_rr_reference(age_years):
    rr_mean_ref = 505 * (age_years ** 0.122)
    if age_years <= 12:
        rr_std_ref = 80 * (age_years ** 0.26)
    else:
        rr_std_ref = 290 * (age_years ** (-0.2))
    return rr_mean_ref, rr_std_ref


def running_stats_init(n_features):
    return {
        "sum": np.zeros(n_features, dtype=np.float64),
        "sumsq": np.zeros(n_features, dtype=np.float64),
        "n": 0,
    }


def running_stats_update(stats, x):
    """
    x: ndarray shape (n_samples, n_features)
    Computes mean/std later with ddof=1.
    """
    if x.size == 0:
        return stats
    x = np.asarray(x, dtype=np.float64)
    stats["sum"] += x.sum(axis=0)
    stats["sumsq"] += np.square(x).sum(axis=0)
    stats["n"] += x.shape[0]
    return stats


def running_stats_finalize(stats, col_names):
    n = stats["n"]
    if n == 0:
        raise ValueError("No samples found while computing interval statistics.")

    mean = stats["sum"] / n

    if n > 1:
        var = (stats["sumsq"] - (stats["sum"] ** 2) / n) / (n - 1)
        var = np.maximum(var, 0.0)
        std = np.sqrt(var)
    else:
        std = np.ones_like(mean)

    std = np.where(std < EPS, 1.0, std)

    return {
        "columns": list(col_names),
        "mean": mean,
        "std": std,
        "n": n,
    }


def normalize_subject_df(df, age_years, interval_mean, interval_std):
    """
    Applies:
    - subject-specific scaling to mean + rr_cols
    - interval z-score to the rest
    """
    df = df.copy()

    rr_mean_ref, rr_std_ref = get_rr_reference(age_years)
    rr_std_ref = max(float(rr_std_ref), EPS)

    # Make sure these columns can store floats
    df[subject_specific_cols] = df[subject_specific_cols].astype(np.float64)
    df[interval_norm_cols] = df[interval_norm_cols].astype(np.float64)

    # Subject-specific normalization
    df.loc[:, subject_specific_cols] = (df[subject_specific_cols] - rr_mean_ref) / rr_std_ref

    # Interval-level z-score normalization
    df.loc[:, interval_norm_cols] = (df[interval_norm_cols] - interval_mean) / interval_std

    return df, rr_mean_ref, rr_std_ref


# ------------------------------------------------------------
# PASS 1: compute interval stats for interval-level features
# ------------------------------------------------------------
interval_stats = {}

for interval, info in subjects_train.items():
    subjects = info["subjects"]

    stats = running_stats_init(len(interval_norm_cols))

    for subj in subjects:
        file_path = Path(subjects_path) / subject_to_filename(subj)
        serie = np.loadtxt(file_path, dtype=int)

        df_temp = extract_hrv_features(
            serie=serie,
            window_size=20,
            window_size_long=40,
        )

        missing = [c for c in interval_norm_cols if c not in df_temp.columns]
        if missing:
            raise KeyError(f"Missing columns in subject {subj}, interval {interval}: {missing}")

        x = df_temp[interval_norm_cols].to_numpy(dtype=np.float64)
        stats = running_stats_update(stats, x)

    interval_stats[interval] = running_stats_finalize(stats, interval_norm_cols)


# ------------------------------------------------------------
# PASS 2: write normalized data progressively to HDF5
# ------------------------------------------------------------
string_dt = h5py.string_dtype(encoding="utf-8")

index_interval = []
index_subject = []
index_path = []
index_n_samples = []
index_age_years = []

with h5py.File(OUTPUT_H5, "w") as h5:
    # Global metadata for reproducibility
    h5.attrs["year_to_weeks"] = YEAR_TO_WEEKS
    h5.attrs["rr_mean_formula"] = "505 * x**0.122"
    h5.attrs["rr_std_formula_le_12"] = "80 * x**0.26"
    h5.attrs["rr_std_formula_gt_12"] = "290 * x**(-0.2)"
    h5.attrs["interval_norm_cols"] = np.array(interval_norm_cols, dtype=string_dt)
    h5.attrs["subject_specific_cols"] = np.array(subject_specific_cols, dtype=string_dt)
    h5.attrs["x_cols"] = np.array(x_cols, dtype=string_dt)
    h5.attrs["y_col"] = y_col

    intervals_group = h5.create_group("intervals")
    norm_group = h5.create_group("normalization")

    for interval, info in subjects_train.items():
        subjects = info["subjects"]

        interval_group = intervals_group.create_group(str(interval))
        norm_interval_group = norm_group.create_group(str(interval))

        # Save interval-level normalization parameters
        stats = interval_stats[interval]
        norm_interval_group.create_dataset("columns", data=np.array(stats["columns"], dtype=string_dt))
        norm_interval_group.create_dataset("mean", data=stats["mean"].astype(np.float64))
        norm_interval_group.create_dataset("std", data=stats["std"].astype(np.float64))
        norm_interval_group.attrs["n_samples"] = int(stats["n"])

        interval_group.attrs["interval"] = str(interval)
        interval_group.attrs["n_subjects"] = len(subjects)
        interval_group.attrs["n_interval_samples"] = 0

        interval_mean = stats["mean"]
        interval_std = np.where(stats["std"] < EPS, 1.0, stats["std"])

        interval_samples_total = 0

        for subj in subjects:
            subj_str = str(subj)
            file_path = Path(subjects_path) / subject_to_filename(subj)

            serie = np.loadtxt(file_path, dtype=int)

            df_temp = extract_hrv_features(
                serie=serie,
                window_size=20,
                window_size_long=40,
            )

            missing = [c for c in (x_cols + [y_col]) if c not in df_temp.columns]
            if missing:
                raise KeyError(f"Missing columns in subject {subj_str}, interval {interval}: {missing}")

            # Identify subject age
            age_years = get_age_years(subj, ages_table, year_to_weeks=YEAR_TO_WEEKS)

            # Normalize
            df_norm, rr_mean_ref, rr_std_ref = normalize_subject_df(
                df_temp,
                age_years=age_years,
                interval_mean=interval_mean,
                interval_std=interval_std,
            )

            # Keep only the requested columns
            X = df_norm[x_cols].to_numpy(dtype=np.float32)
            y = df_norm[y_col].to_numpy()

            n_samples = X.shape[0]
            interval_samples_total += n_samples

            # Create subject group
            subject_group = interval_group.create_group(f"subject_{subj_str}")
            subject_group.create_dataset("X", data=X, compression="gzip", compression_opts=4, chunks=True)
            subject_group.create_dataset("y", data=y, compression="gzip", compression_opts=4, chunks=True)

            # Subject identification and reproducibility metadata
            subject_group.attrs["file_id"] = subj_str
            subject_group.attrs["interval"] = str(interval)
            subject_group.attrs["age_years"] = float(age_years)
            subject_group.attrs["rr_mean_ref"] = float(rr_mean_ref)
            subject_group.attrs["rr_std_ref"] = float(rr_std_ref)
            subject_group.attrs["n_samples"] = int(n_samples)
            subject_group.attrs["x_cols"] = np.array(x_cols, dtype=string_dt)
            subject_group.attrs["y_col"] = y_col

            # Index table for proportional sampling
            index_interval.append(str(interval))
            index_subject.append(subj_str)
            index_path.append(f"/intervals/{interval}/subject_{subj_str}")
            index_n_samples.append(int(n_samples))
            index_age_years.append(float(age_years))

        interval_group.attrs["n_interval_samples"] = int(interval_samples_total)

    # Save the index table at the end
    index_group = h5.create_group("index")
    index_group.create_dataset("interval", data=np.array(index_interval, dtype=string_dt))
    index_group.create_dataset("subject_id", data=np.array(index_subject, dtype=string_dt))
    index_group.create_dataset("h5_path", data=np.array(index_path, dtype=string_dt))
    index_group.create_dataset("n_samples", data=np.array(index_n_samples, dtype=np.int64))
    index_group.create_dataset("age_years", data=np.array(index_age_years, dtype=np.float64))

    # Convenience totals
    total_samples = int(np.sum(index_n_samples))
    index_group.attrs["total_samples"] = total_samples

print(f"Saved HDF5 dataset to: {OUTPUT_H5}")

Saved HDF5 dataset to: hrv_dataset.h5
